# Improved Collaborative Filtering for Colab

This notebook is a separate replacement/experiment file for the Collaborative Filtering part. It does not modify the old notebook.

Main changes:
- denser k-core filtering (`user >= 10`, `item >= 10` by default)
- leave-one-positive-out or time-based split per user
- explicit ALS and implicit ALS experiments
- top-K evaluation after excluding train items
- metrics: Precision@K, Recall@K, NDCG@K, MAP, HitRate@K


In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.1"])

from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path

# Change these paths for your Colab/Drive layout.
DATA_PATH = "/content/drive/MyDrive/RS/rating_rs.parquet"
OUTPUT_DIR = "/content/drive/MyDrive/RS/cf_improved_outputs"

# Filtering: make the matrix denser than the original >=5 setting.
MIN_USER_RATINGS = 10
MIN_ITEM_RATINGS = 10
KCORE_ITERATIONS = 3

# Split options: "time" keeps the latest positive item per user as test.
# "random" keeps one random positive item per user as test.
SPLIT_MODE = "time"
RELEVANT_THRESHOLD = 4.0

# Candidate pool must be larger than K because train items are removed before evaluation.
K_VALUES = [10, 20]
CANDIDATE_K = 200
OUTPUT_TOP_K = 20

# Keep grids small first. Expand after the pipeline runs successfully.
EXPLICIT_CONFIGS = [
    {"rank": 20, "regParam": 0.1, "maxIter": 10},
    {"rank": 50, "regParam": 0.1, "maxIter": 10},
    {"rank": 50, "regParam": 0.2, "maxIter": 15},
]

IMPLICIT_CONFIGS = [
    {"rank": 20, "regParam": 0.1, "maxIter": 10, "alpha": 10.0},
    {"rank": 50, "regParam": 0.1, "maxIter": 10, "alpha": 20.0},
    {"rank": 50, "regParam": 0.2, "maxIter": 15, "alpha": 20.0},
]

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [3]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    avg,
    col,
    collect_set,
    count,
    desc,
    expr,
    lit,
    max as spark_max,
    rand,
    row_number,
    size,
    array_intersect,
    when,
)
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.mllib.evaluation import RankingMetrics

spark = (
    SparkSession.builder
    .appName("improved-collaborative-filtering")
    .config("spark.sql.shuffle.partitions", "96")
    .config("spark.driver.memory", "12g")
    .config("spark.executor.memory", "12g")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/content/spark_checkpoints")
print(spark.version)


4.0.2


## Load data and create a denser training set

The original project uses at least 5 ratings per user/item. This experiment uses a denser threshold by default. A denser matrix usually helps ALS learn better latent factors, but reduces catalog/user coverage.


In [4]:
ratings_raw = spark.read.parquet(DATA_PATH)
ratings_raw.printSchema()

required_cols = ["user_id", "product_id", "review_rating"]
missing = [c for c in required_cols if c not in ratings_raw.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

date_col = "review_date" if "review_date" in ratings_raw.columns else None
select_cols = required_cols + ([date_col] if date_col else [])

ratings = (
    ratings_raw
    .select(*select_cols)
    .dropna(subset=required_cols)
    .withColumn("review_rating", col("review_rating").cast("double"))
)

if date_col:
    ratings = ratings.withColumn("review_date", col("review_date").cast("long"))
else:
    ratings = ratings.withColumn("review_date", lit(0).cast("long"))

# One row per user-item pair. If duplicates exist, keep average rating and latest timestamp.
ratings = (
    ratings
    .groupBy("user_id", "product_id")
    .agg(
        avg("review_rating").alias("review_rating"),
        spark_max("review_date").alias("review_date"),
    )
)

print("Raw user-item rows:", ratings.count())
ratings.show(5, truncate=False)


root
 |-- product_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- review_rating: double (nullable = true)
 |-- review_date: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

Raw user-item rows: 2128558
+----------------------------+----------+-------------+-------------+
|user_id                     |product_id|review_rating|review_date  |
+----------------------------+----------+-------------+-------------+
|AGACHNBMLRDXMGASGKH6TDTE6S6A|0060883286|4.0          |1523210361251|
|AH44KOLWBOSIK4LCLTQGZRQ4NHTQ|0060883286|5.0          |1548258059056|
|AEPTGOALOMDYRICG2PYKRZUGVROQ|0528012770|5.0          |1498227890698|
|AE5O57G3HEDWC4H552MTM7EZT5EA|0545019222|4.0          |1361911569000|
|AGGSTW45STYFURWJDYELIS3PEB3A|0545056578|5.0          |1491382616000|
+----------------------------+----------+-------------+-------------+
only showing top 5 rows


In [5]:
def apply_kcore(df, min_user_ratings=10, min_item_ratings=10, iterations=3):
    out = df
    for i in range(iterations):
        user_counts = out.groupBy("user_id").agg(count("*").alias("user_count"))
        out = out.join(user_counts.filter(col("user_count") >= min_user_ratings).select("user_id"), "user_id")

        item_counts = out.groupBy("product_id").agg(count("*").alias("item_count"))
        out = out.join(item_counts.filter(col("item_count") >= min_item_ratings).select("product_id"), "product_id")
        out = out.checkpoint(eager=True)
        print(f"k-core iteration {i + 1}: {out.count():,} rows")
    return out

ratings_dense = apply_kcore(
    ratings,
    min_user_ratings=MIN_USER_RATINGS,
    min_item_ratings=MIN_ITEM_RATINGS,
    iterations=KCORE_ITERATIONS,
)

summary = {
    "rows": ratings_dense.count(),
    "users": ratings_dense.select("user_id").distinct().count(),
    "items": ratings_dense.select("product_id").distinct().count(),
}
summary


k-core iteration 1: 400,299 rows
k-core iteration 2: 162,577 rows
k-core iteration 3: 89,160 rows


{'rows': 89160, 'users': 7236, 'items': 4449}

## Index IDs and split by user

For top-K evaluation, each evaluated user needs known history in train and at least one relevant item in test. This notebook keeps one positive item per user as test, using either latest timestamp or random selection.


In [6]:
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_idx_raw", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_idx_raw", handleInvalid="skip")
index_pipeline = Pipeline(stages=[user_indexer, item_indexer])
index_model = index_pipeline.fit(ratings_dense)

ratings_indexed = (
    index_model
    .transform(ratings_dense)
    .withColumn("user_idx", col("user_idx_raw").cast("int"))
    .withColumn("item_idx", col("item_idx_raw").cast("int"))
    .select("user_id", "product_id", "user_idx", "item_idx", "review_rating", "review_date")
    .checkpoint(eager=True)
)

print("Indexed rows:", ratings_indexed.count())
ratings_indexed.show(5, truncate=False)


Indexed rows: 89160
+----------------------------+----------+--------+--------+-------------+-------------+
|user_id                     |product_id|user_idx|item_idx|review_rating|review_date  |
+----------------------------+----------+--------+--------+-------------+-------------+
|AEJ5QBDRY75TXAH5R476VHPU7PRA|B003H2GBM4|600     |17      |5.0          |1470964637000|
|AEUD4LXIHBZEL5EFCU2PFMMEWNYQ|B00006IC89|5811    |33      |5.0          |1493692646000|
|AEVY2V4QQW3ANKZVIJPVUALC55YA|B003H2GBM4|4791    |17      |5.0          |1479909767000|
|AEVY2V4QQW3ANKZVIJPVUALC55YA|B00FW1N1IU|4791    |2194    |5.0          |1461591021000|
|AFLBI3UPU6LXL4NUKTCVNKOKJBFA|B001GXHG2U|1089    |1347    |5.0          |1447232540000|
+----------------------------+----------+--------+--------+-------------+-------------+
only showing top 5 rows


In [7]:
def split_one_positive_per_user(df, mode="time", relevant_threshold=4.0):
    positives = df.filter(col("review_rating") >= relevant_threshold)

    if mode == "time":
        w = Window.partitionBy("user_idx").orderBy(desc("review_date"), desc("review_rating"), col("item_idx"))
    elif mode == "random":
        positives = positives.withColumn("_rand", rand(seed=42))
        w = Window.partitionBy("user_idx").orderBy(col("_rand"))
    else:
        raise ValueError("SPLIT_MODE must be 'time' or 'random'")

    test_keys = (
        positives
        .withColumn("rn", row_number().over(w))
        .filter(col("rn") == 1)
        .select("user_idx", "item_idx")
        .distinct()
    )

    test = df.join(test_keys, ["user_idx", "item_idx"], "inner").checkpoint(eager=True)
    train = df.join(test_keys, ["user_idx", "item_idx"], "left_anti").checkpoint(eager=True)

    # Keep only users with train history and test ground truth.
    train_users = train.select("user_idx").distinct()
    test_users = test.select("user_idx").distinct()
    eval_users = train_users.join(test_users, "user_idx", "inner").checkpoint(eager=True)
    train = train.join(eval_users, "user_idx", "inner").checkpoint(eager=True)
    test = test.join(eval_users, "user_idx", "inner").checkpoint(eager=True)

    return train, test, eval_users

train, test, eval_users = split_one_positive_per_user(
    ratings_indexed,
    mode=SPLIT_MODE,
    relevant_threshold=RELEVANT_THRESHOLD,
)

print("Train rows:", train.count())
print("Test rows:", test.count())
print("Eval users:", eval_users.count())


Train rows: 81907
Test rows: 7176
Eval users: 7176


## Evaluation helpers

`recommendForUserSubset` may return items already seen in train. For offline top-K metrics, those train items are removed first, then the final top-K list is compared with test positives.


In [8]:
def topk_rows_after_excluding_train(model, train_df, eval_users_df, k, candidate_k=200):
    recs = model.recommendForUserSubset(eval_users_df.select("user_idx").distinct(), candidate_k)
    exploded = (
        recs
        .select("user_idx", expr("explode(recommendations) as rec"))
        .select("user_idx", col("rec.item_idx").cast("int").alias("item_idx"), col("rec.rating").alias("score"))
    )

    train_items = train_df.select("user_idx", "item_idx").distinct()
    filtered = exploded.join(train_items, ["user_idx", "item_idx"], "left_anti")

    w = Window.partitionBy("user_idx").orderBy(desc("score"))
    return (
        filtered
        .withColumn("rank", row_number().over(w))
        .filter(col("rank") <= k)
        .checkpoint(eager=True)
    )


def topk_lists_after_excluding_train(model, train_df, eval_users_df, k, candidate_k=200):
    rows = topk_rows_after_excluding_train(model, train_df, eval_users_df, k, candidate_k)
    return (
        rows
        .groupBy("user_idx")
        .agg(expr("transform(array_sort(collect_list(struct(rank, item_idx))), x -> x.item_idx)").alias("predicted_items"))
    )


def evaluate_topk(model, train_df, test_df, eval_users_df, k=10, candidate_k=200):
    predicted = topk_lists_after_excluding_train(model, train_df, eval_users_df, k, candidate_k)
    ground_truth = (
        test_df
        .filter(col("review_rating") >= RELEVANT_THRESHOLD)
        .groupBy("user_idx")
        .agg(collect_set("item_idx").alias("relevant_items"))
    )

    joined = predicted.join(ground_truth, "user_idx", "inner").filter(size(col("relevant_items")) > 0)
    n_users = joined.count()
    if n_users == 0:
        return {"k": k, "eval_users": 0, "precision": 0.0, "recall": 0.0, "ndcg": 0.0, "map": 0.0, "hit_rate": 0.0}

    rdd = joined.rdd.map(lambda r: (r.predicted_items, r.relevant_items))
    metrics = RankingMetrics(rdd)

    recall_hit = (
        joined
        .withColumn("hits", size(array_intersect(col("predicted_items"), col("relevant_items"))))
        .withColumn("recall", col("hits") / size(col("relevant_items")))
        .withColumn("hit", when(col("hits") > 0, lit(1.0)).otherwise(lit(0.0)))
        .agg(avg("recall").alias("recall"), avg("hit").alias("hit_rate"))
        .collect()[0]
    )

    return {
        "k": k,
        "eval_users": n_users,
        "precision": float(metrics.precisionAt(k)),
        "recall": float(recall_hit["recall"]),
        "ndcg": float(metrics.ndcgAt(k)),
        "map": float(metrics.meanAveragePrecision),
        "hit_rate": float(recall_hit["hit_rate"]),
    }


## Explicit ALS

Explicit ALS predicts ratings. It is useful for RMSE, but top-K quality should still be selected by ranking metrics such as NDCG@K or Recall@K.


In [9]:
rmse_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="review_rating",
    predictionCol="prediction",
)

experiment_rows = []
trained_models = {}

for cfg in EXPLICIT_CONFIGS:
    name = f"explicit_rank{cfg['rank']}_reg{cfg['regParam']}_iter{cfg['maxIter']}"
    print("Training", name)
    als = ALS(
        userCol="user_idx",
        itemCol="item_idx",
        ratingCol="review_rating",
        coldStartStrategy="drop",
        nonnegative=True,
        implicitPrefs=False,
        rank=cfg["rank"],
        regParam=cfg["regParam"],
        maxIter=cfg["maxIter"],
        seed=42,
    )
    model = als.fit(train)
    predictions = model.transform(test).dropna(subset=["prediction"])
    rmse = rmse_evaluator.evaluate(predictions) if predictions.count() else None
    trained_models[name] = (model, "explicit", cfg)

    for k in K_VALUES:
        metrics = evaluate_topk(model, train, test, eval_users, k=k, candidate_k=CANDIDATE_K)
        metrics.update({"model": name, "type": "explicit", "rmse": rmse, **cfg})
        experiment_rows.append(metrics)
        print(metrics)


Training explicit_rank20_reg0.1_iter10


/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


{'k': 10, 'eval_users': 7176, 'precision': 0.00020903010033444848, 'recall': 0.0020903010033444815, 'ndcg': 0.0009279104129925292, 'map': 0.0005803626727539775, 'hit_rate': 0.0020903010033444815, 'model': 'explicit_rank20_reg0.1_iter10', 'type': 'explicit', 'rmse': 0.8770497175978054, 'rank': 20, 'regParam': 0.1, 'maxIter': 10}
{'k': 20, 'eval_users': 7176, 'precision': 0.0001950947603121519, 'recall': 0.0039018952062430325, 'ndcg': 0.0013927496955139281, 'map': 0.0007119773110575755, 'hit_rate': 0.0039018952062430325, 'model': 'explicit_rank20_reg0.1_iter10', 'type': 'explicit', 'rmse': 0.8770497175978054, 'rank': 20, 'regParam': 0.1, 'maxIter': 10}
Training explicit_rank50_reg0.1_iter10
{'k': 10, 'eval_users': 7176, 'precision': 0.0003762541806020069, 'recall': 0.003762541806020067, 'ndcg': 0.0017551298543912323, 'map': 0.0011641538815451865, 'hit_rate': 0.003762541806020067, 'model': 'explicit_rank50_reg0.1_iter10', 'type': 'explicit', 'rmse': 0.8520129199772566, 'rank': 50, 'regPar

## Implicit ALS

Implicit ALS treats positive ratings as interaction signals. This usually aligns better with top-K recommendation metrics than rating-prediction RMSE.


In [10]:
implicit_train = (
    train
    .withColumn("implicit_strength", when(col("review_rating") >= RELEVANT_THRESHOLD, lit(1.0)).otherwise(lit(0.0)))
    .filter(col("implicit_strength") > 0)
    .checkpoint(eager=True)
)

print("Implicit train rows:", implicit_train.count())

for cfg in IMPLICIT_CONFIGS:
    name = f"implicit_rank{cfg['rank']}_reg{cfg['regParam']}_iter{cfg['maxIter']}_alpha{cfg['alpha']}"
    print("Training", name)
    als = ALS(
        userCol="user_idx",
        itemCol="item_idx",
        ratingCol="implicit_strength",
        coldStartStrategy="drop",
        nonnegative=True,
        implicitPrefs=True,
        rank=cfg["rank"],
        regParam=cfg["regParam"],
        maxIter=cfg["maxIter"],
        alpha=cfg["alpha"],
        seed=42,
    )
    model = als.fit(implicit_train)
    trained_models[name] = (model, "implicit", cfg)

    for k in K_VALUES:
        metrics = evaluate_topk(model, train, test, eval_users, k=k, candidate_k=CANDIDATE_K)
        metrics.update({"model": name, "type": "implicit", "rmse": None, **cfg})
        experiment_rows.append(metrics)
        print(metrics)


Implicit train rows: 72723
Training implicit_rank20_reg0.1_iter10_alpha10.0
{'k': 10, 'eval_users': 7155, 'precision': 0.002907058001397621, 'recall': 0.02907058001397624, 'ndcg': 0.015767738961379323, 'map': 0.0118216809202134, 'hit_rate': 0.02907058001397624, 'model': 'implicit_rank20_reg0.1_iter10_alpha10.0', 'type': 'implicit', 'rmse': None, 'rank': 20, 'regParam': 0.1, 'maxIter': 10, 'alpha': 10.0}
{'k': 20, 'eval_users': 7155, 'precision': 0.0025227113906359047, 'recall': 0.050454227812718376, 'ndcg': 0.021142230607489737, 'map': 0.013278729066401623, 'hit_rate': 0.050454227812718376, 'model': 'implicit_rank20_reg0.1_iter10_alpha10.0', 'type': 'implicit', 'rmse': None, 'rank': 20, 'regParam': 0.1, 'maxIter': 10, 'alpha': 10.0}
Training implicit_rank50_reg0.1_iter10_alpha20.0
{'k': 10, 'eval_users': 7155, 'precision': 0.002948986722571632, 'recall': 0.029489867225716282, 'ndcg': 0.01399641242810282, 'map': 0.009345889765176958, 'hit_rate': 0.029489867225716282, 'model': 'implicit_

## Compare and save outputs

The best model below is selected by NDCG at the largest K in `K_VALUES`. You can switch the selection metric to `recall` or `hit_rate` if your report prioritizes coverage over rank quality.


In [11]:
import pandas as pd

results_pdf = pd.DataFrame(experiment_rows)
results_pdf = results_pdf.sort_values(["k", "ndcg", "recall", "precision"], ascending=[True, False, False, False])
display(results_pdf)

metrics_path = str(Path(OUTPUT_DIR) / "cf_improved_metrics.csv")
results_pdf.to_csv(metrics_path, index=False)
print("Saved metrics:", metrics_path)

target_k = max(K_VALUES)
best_row = (
    results_pdf[results_pdf["k"] == target_k]
    .sort_values(["ndcg", "recall", "hit_rate"], ascending=False)
    .iloc[0]
)

best_name = best_row["model"]
best_model, best_type, best_cfg = trained_models[best_name]
print("Best model:", best_name)
print(best_row.to_dict())


,k,eval_users,precision,recall,ndcg,map,hit_rate,model,type,rmse,rank,regParam,maxIter,alpha
10,10,7155,0.003536,0.035360,0.016892,0.011329,0.035360,implicit_rank50_reg0.2_iter15_alpha20.0,implicit,NaN,50,0.2,15,20.0
6,10,7155,0.002907,0.029071,0.015768,0.011822,0.029071,implicit_rank20_reg0.1_iter10_alpha10.0,implicit,NaN,20,0.1,10,10.0
8,10,7155,0.002949,0.029490,0.013996,0.009346,0.029490,implicit_rank50_reg0.1_iter10_alpha20.0,implicit,NaN,50,0.1,10,20.0
4,10,7176,0.000348,0.003484,0.001761,0.001250,0.003484,explicit_rank50_reg0.2_iter15,explicit,0.881630,50,0.2,15,NaN
2,10,7176,0.000376,0.003763,0.001755,0.001164,0.003763,explicit_rank50_reg0.1_iter10,explicit,0.852013,50,0.1,10,NaN
0,10,7176,0.000209,0.002090,0.000928,0.000580,0.002090,explicit_rank20_reg0.1_iter10,explicit,0.877050,20,0.1,10,NaN
11,20,7155,0.002676,0.053529,0.021456,0.012567,0.053529,implicit_rank50_reg0.2_iter15_alpha20.0,implicit,NaN,50,0.2,15,20.0
7,20,7155,0.002523,0.050454,0.021142,0.013279,0.050454,implicit_rank20_reg0.1_iter10_alpha10.0,implicit,NaN,20,0.1,10,10.0
9,20,7155,0.002369,0.047379,0.018506,0.010576,0.047379,implicit_rank50_reg0.1_iter10_alpha20.0,implicit,NaN,50,0.1,10,20.0
5,20,7176,0.000334,0.006689,0.002551,0.001456,0.006689,explicit_rank50_reg0.2_iter15,explicit,0.881630,50,0.2,15,NaN


Saved metrics: /content/drive/MyDrive/RS/cf_improved_outputs/cf_improved_metrics.csv
Best model: implicit_rank50_reg0.2_iter15_alpha20.0
{'k': 20, 'eval_users': 7155, 'precision': 0.0026764500349406055, 'recall': 0.05352900069881202, 'ndcg': 0.02145561546512037, 'map': 0.012566660342263832, 'hit_rate': 0.05352900069881202, 'model': 'implicit_rank50_reg0.2_iter15_alpha20.0', 'type': 'implicit', 'rmse': nan, 'rank': 50, 'regParam': 0.2, 'maxIter': 15, 'alpha': 20.0}


In [12]:
best_rows = topk_rows_after_excluding_train(
    best_model,
    train,
    eval_users,
    k=OUTPUT_TOP_K,
    candidate_k=max(CANDIDATE_K, OUTPUT_TOP_K * 10),
)

user_labels = index_model.stages[0].labels
item_labels = index_model.stages[1].labels

idx_to_user = spark.createDataFrame([(int(i), v) for i, v in enumerate(user_labels)], ["user_idx", "user_id"])
idx_to_item = spark.createDataFrame([(int(i), v) for i, v in enumerate(item_labels)], ["item_idx", "product_id"])

decoded_recs = (
    best_rows
    .join(idx_to_user, "user_idx", "inner")
    .join(idx_to_item, "item_idx", "inner")
    .select("user_id", "product_id", col("score").alias("cf_score"), "rank")
)

recs_path = str(Path(OUTPUT_DIR) / "cf_improved_recommendations.parquet")
decoded_recs.write.mode("overwrite").parquet(recs_path)
print("Saved recommendations:", recs_path)
decoded_recs.show(20, truncate=False)


Saved recommendations: /content/drive/MyDrive/RS/cf_improved_outputs/cf_improved_recommendations.parquet
+----------------------------+----------+----------+----+
|user_id                     |product_id|cf_score  |rank|
+----------------------------+----------+----------+----+
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B0006HVKOC|0.77986944|20  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B005VC8CG6|0.79608595|19  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B000J09CO6|0.80495703|18  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B00E58RQ6Y|0.8121412 |17  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B001CSMJJE|0.8179649 |16  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B000NNXY8Y|0.8208554 |15  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B0027JIIKQ|0.8316334 |14  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B000078CQ7|0.8612946 |13  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B002766V3K|0.8712203 |12  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B00004Z64M|0.91387165|11  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B002VD6BLG|0.93261683|10  |
|AFYQNJXE63KGPUADMLCIZNYBZZGA|B0003WN0CA|0.9376851 |9   |
|AFYQNJXE63KGPUADMLCIZNYB

## What to report

Use this wording if the improved version gives better ranking metrics:

> The original ALS pipeline used random 80/20 split and evaluated top-K directly. The improved CF experiment uses denser k-core filtering, one-positive-out user-level split, implicit ALS, and removes train items before top-K evaluation. This makes the offline evaluation closer to a real recommendation scenario and usually improves ranking metrics such as Recall@K, NDCG@K, and HitRate@K.

If metrics are still low, explain that Amazon recommendation is sparse and has a very large item catalog, so exact top-K hits are hard. Then compare ALS against popular baseline or hybrid instead of only reporting raw ALS metrics.
